# Topic 20c - Merging and the four join types

Practice for section 4 of the [main lesson](topic20_data_wrangling.ipynb).

`merge` looks up rows in one table using a key found in another. Before you write one, answer three questions:

1. **Which key?** The column whose values appear in both tables.
2. **Which rows do I keep?** The `how=` argument.
3. **How many rows should I have afterwards?** Decide before running, then check.

Question 3 is the one people skip, and it is the one that catches mistakes.

In [ ]:
from pathlib import Path

import pandas as pd


def find_data_dir():
    """Locate the lesson's data folder from any sensible working directory."""
    for candidate in [Path.cwd() / "data",
                      Path.cwd() / "data_wrangling" / "data",
                      Path.cwd().parent / "data"]:
        if (candidate / "orders.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find orders.csv - check your data folder.")


DATA_DIR = find_data_dir()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)

orders = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=["order_date"])
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
regions = pd.read_csv(DATA_DIR / "regions.csv")
returns = pd.read_csv(DATA_DIR / "returns.csv")
orders["revenue_ghs"] = orders["quantity"] * orders["unit_price_ghs"]
print("loaded:", {"orders": len(orders), "customers": len(customers),
                  "products": len(products), "regions": len(regions),
                  "returns": len(returns)})

## 1. The four joins on a table you can check by eye

Three orders and three customers, arranged so that each join gives a different answer.

In [ ]:
small_orders = pd.DataFrame({
    "order_id": ["O1", "O2", "O3"],
    "customer_id": ["C1", "C2", "C9"],       # C9 is unknown
    "amount_ghs": [120, 340, 75],
})
small_customers = pd.DataFrame({
    "customer_id": ["C1", "C2", "C3"],       # C3 has never ordered
    "customer_name": ["Kwame Mensah", "Ama Osei", "Yaw Boateng"],
})

print("ORDERS:");    print(small_orders.to_string(index=False))
print("\nCUSTOMERS:"); print(small_customers.to_string(index=False))

In [ ]:
for how in ["inner", "left", "right", "outer"]:
    result = small_orders.merge(small_customers, on="customer_id", how=how)
    print(f"\n{'=' * 52}\nhow='{how}'   ->   {len(result)} rows\n{'=' * 52}")
    print(result.to_string(index=False))

### What each answer means

| `how=` | Rows | What happened |
|---|---|---|
| `inner` | 2 | Order `O3` **vanished**. Customer `C9` is unknown, so the row was dropped with no warning. |
| `left` | 3 | `O3` survives, with `customer_name` as `NaN`. The problem is now visible. |
| `right` | 3 | Customer `C3` appears with no order. Useful for "who has never bought from us?" |
| `outer` | 4 | Everything from both sides. The reconciliation view. |

**`left` is an analyst's default.** Your fact table records things that happened; losing rows because a lookup is incomplete is a data problem to investigate, not a tidying-up to accept.

### Try It - predict before you run

Before running the next cell, write down how many rows you expect from each join. Then check.

In [ ]:
a = pd.DataFrame({"key": ["x", "y", "z"], "left_value": [1, 2, 3]})
b = pd.DataFrame({"key": ["y", "z", "w", "v"], "right_value": [10, 20, 30, 40]})

# YOUR PREDICTIONS:
# inner: ___   left: ___   right: ___   outer: ___

for how in ["inner", "left", "right", "outer"]:
    print(f"{how:>6}: {len(a.merge(b, on='key', how=how))} rows")

## 2. Merging the real tables

Now the real thing. `orders` is the fact table; `customers` and `products` are lookups.

In [ ]:
with_customers = orders.merge(customers, on="customer_id", how="left")
print(f"orders:         {len(orders)} rows")
print(f"after merge:    {len(with_customers)} rows")
print(f"row count kept: {len(with_customers) == len(orders)}")
print(f"\nNew columns gained: "
      f"{[c for c in with_customers.columns if c not in orders.columns]}")
print()
print(with_customers[["order_id", "customer_id", "customer_name", "region",
                      "revenue_ghs"]].head(4).to_string(index=False))

## 3. When the key has different names

Real tables rarely agree on column names. `left_on` and `right_on` handle that.

In [ ]:
renamed = customers.rename(columns={"customer_id": "id"})

joined = orders.merge(renamed, left_on="customer_id", right_on="id", how="left")
print(f"merged on differently-named keys: {len(joined)} rows")
print(f"Both key columns are kept: {'customer_id' in joined.columns and 'id' in joined.columns}")

# Drop the redundant copy afterwards.
joined = joined.drop(columns="id")
print(f"After dropping the duplicate key column: {len(joined.columns)} columns")

## 4. Merging on more than one key

Sometimes a row is only identified by a combination. Pass a list.

In [ ]:
monthly_actuals = (orders.assign(month=orders["order_date"].dt.strftime("%b"))
                         .merge(customers, on="customer_id", how="left")
                         .groupby(["region", "month"], as_index=False)["revenue_ghs"]
                         .sum())
print(f"One row per region per month: {monthly_actuals.shape}")
print(monthly_actuals.head(4).to_string(index=False))
print("\nA merge on ['region', 'month'] would match these correctly.")
print("A merge on 'region' alone would match each region's actuals to ALL")
print("twelve of its targets and produce twelve times too many rows.")

## 5. Practice

1. Merge `products` onto `orders` with a **left** join and report the row count. (It will surprise you - notebook 20d explains why.)
2. Merge `regions` onto `customers` with a left join. How many customers end up with no `zone`?
3. Use an **inner** join between `orders` and `customers`, and work out how many orders were lost and what they were worth.
4. Use an **outer** join between `orders` and `customers` with `indicator=True`, and count how many customers have never placed an order.

In [ ]:
# YOUR CODE HERE

---
## Where this fits

- [Back to the main lesson](topic20_data_wrangling.ipynb)
- Previous: [20b - Stacking with concat](topic20b_stacking_with_concat.ipynb)
- Next: [20d - Merge hazards and validation](topic20d_merge_hazards_and_validation.ipynb)